In [ ]:
import pandas as pd
import spacy
from spacy.language import Language
from spacy.tokens import Span
from thefuzz import fuzz, process
from tqdm.auto import tqdm
from newspaper import Article, Config
from newspaper.article import ArticleException
from selenium import webdriver
from selenium.webdriver import ChromeOptions
import threading
from dotenv import load_dotenv
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from thefuzz import fuzz, process
from NewsSentiment import TargetSentimentClassifier
import numpy as np
from typing import Literal
tqdm.pandas()

In [2]:
load_dotenv()
serp_api_key = os.environ.get('SERP_API_KEY')

def get_selenium_html(url):
    with threading.Lock():
        with webdriver.Chrome(options=options) as driver: 
            driver.get(url)
            article_html = driver.page_source
        return article_html

options = ChromeOptions()
options.page_load_strategy = 'eager'
options.add_argument("--headless=new")

config = Config()
config.browser_user_agent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124  Safari/537.36'
config.request_timeout = 20

In [3]:
def parse_url(url) -> str | None:
    try:
        article = Article(url)
        article.download()
        article.parse()

        text = " ".join(article.text.split()[:200])
        if text == '':
            raise LookupError(f"First Attempt: Unable to extact article text for {url}. Retrying...") # if fail, go into second try

        return text
    
    except (ArticleException, LookupError) as e1:
        try:
            if str(e1).find('403') != -1 or isinstance(e1, LookupError):
                article = Article(url, config=config)
                article.download()
                article.parse()

                text = " ".join(article.text.split()[:200])
                if text == '':
                    raise LookupError(f"Second Attempt: Unable to extact article text for {url}. Retrying...") # if fail, go into third try
                else:
                    print(f'Retry succeeded for {url}!')
                
                return text
            
            else:
                print(e1)
                return ''
            
        except (ArticleException, LookupError) as e2:
            if str(e2).find('403') != -1 or isinstance(e2, LookupError):
                article = Article(url, config=config)
                article.download(input_html = get_selenium_html(url))
                article.parse()
                text = " ".join(article.text.split()[:200])
                if text == '':
                    text = ''
                    print(f"Final Attempt: Unable to extact article text for {url}") # if it still fails, can't circumvent.
                else:
                    print(f'Retry succeeded for {url}!')
                
                return text

            else:
                print(e2)
                return ''


def parse_stories_parallel(df, max_threads=80):

    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        results = list(tqdm(executor.map(parse_url, df['url']), total=len(df)))

    df['text'] = results
    df['text'] = df['text'].replace('', pd.NA)
    
    return df

In [ ]:
df_articles = pd.read_csv('elections_opinions.csv')#.sample(10, random_state=42)

df_articles

In [ ]:
df_articles = parse_stories_parallel(df_articles)
df_articles = df_articles.dropna(subset='text').copy()
df_articles

In [5]:
df_corpus = pd.read_json('nivaduck_with_display_names.json')
df_corpus = df_corpus.dropna(subset='display_name')
corpus = df_corpus['display_name'].to_dict()

In [6]:
EST = {
    'BJP',
    'AIADMK',
    #'GOV',
    'NCP',
    'ABVP',
    'TDP',
    'AMMK',
    'JDU',
    'RSS',
    'JDS',
    'MNS',
    'LJP',
    'VHP'
}

OPP = {
    'INC',
    'DMK',
    'AAP',
    'SP',
    'SAMAJWADI',
    'SAMAJWADI ',
    'BJD',
    'TRS',
    'CPIM',
    'AITC',
    'TMC',
    'RJD',
    'AIMIM',
    'JKNC',
    'BSP',
    'JKPDP',
    'JMM',
    'CPIML'
}

In [7]:
nlp = spacy.load('en_core_web_trf')
@Language.component("fuzzy_affiliation")
def fuzzy_affiliates(doc):
    ents = []
    
    for ent in doc.ents:
        if ent.label_ not in {"PERSON", "ORG"}:
            ents.append(ent)
            continue
        
        match = process.extractOne(ent.text, corpus, scorer=fuzz.ratio, score_cutoff=95)

        if not match:
            ents.append(ent)
            continue

        _, _, index =  match
        party = str(df_corpus.loc[index, 'party'])

        if party in EST:
            aff = 'EST'
        elif party in OPP:
            aff = 'OPP'
        else:
            ents.append(ent)
            continue
        
        new_ent = Span(doc, ent.start, ent.end, label=aff)
        ents.append(new_ent)
            
    doc.ents = ents
    return doc

nlp.add_pipe("fuzzy_affiliation", after='ner')

<function __main__.fuzzy_affiliates(doc)>

In [8]:
newsmtsc_classifier = TargetSentimentClassifier()

In [9]:
def batch_nlp(df, entity_type:Literal['EST', 'OPP'], batch_size=16):
    #print("[Running NER...]")
    data = []
    texts = list(df['text'])
    #for text in texts: print(text, end='\n\n')
    story_datapoints_tracker = [] # for the story at the i'th index, how many newsmtsc tuples it has
    
    for doc in tqdm(nlp.pipe(texts, batch_size=batch_size), total=len(texts), desc="[Running NER...]"):
        
        data_count = 0

        for ent in doc.ents:
            if ent.label_ == entity_type:
                sentence = ent.sent
                left = doc.text[sentence.start_char : ent.start_char]
                entity_str = ent.text
                right= doc.text[ent.end_char : sentence.end_char]

                left = left[-250:] if len(left) > 250 else left
                right = right[:250] if len(right) > 250 else right
                # handling TooLongTextException - sometimes Newspaper4k won't parse things correctly
                # and you'll end up with the sentence a size of a full article, which newsMTSC will 
                # probably explode on. Hence, truncate

                data.append((left, entity_str, right))
                data_count += 1
        
        story_datapoints_tracker.append(data_count)
    
    if data:
        print("[Running sentiments analysis...]")
        sentiments = newsmtsc_classifier.infer(targets=data, batch_size=batch_size)
        return data, story_datapoints_tracker, sentiments
    
    else:

        print("Could not find ANY relavant entities!")
        return [], [], []


def label_stories(df, entity_type:Literal['EST', 'OPP'],
                  data, story_datapoints_tracker, sentiments):
    print()
    
    label = f'{entity_type}_label'
    ent_col = f'{entity_type}_ents'
    if data == []:
        df[ent_col] = pd.NA
        df[label] = 'unknown'
        return df
    
    labels = []
    all_ents = []
    i = 0

    for k in story_datapoints_tracker:

        vectors = []
        story_ents = set()

        for datapoint, sentiment in zip(data[i:i+k],sentiments[i:i+k]):
            story_ents.add(datapoint[1])
            probs = {item['class_label']: item['class_prob'] for item in sentiment}

            if max(probs.values()) < 0.75: continue

            vector = [probs['positive'], probs['neutral'], probs['negative']]
            print(datapoint)
            print(vector)
            vectors.append(vector)
            
        
        all_ents.append(story_ents)
        
        if not vectors:
            labels.append('unknown')
            i += k
            continue

        print()

        vectors = np.array(vectors)
        article_vec = np.mean(vectors, axis=0)
        #print('Outlet:', story['outlet'])
        #print('URL:', story['url'])
        print('Article Vector:', article_vec)
        class_index = np.argmax(article_vec)

        if class_index == 0:
            labels.append('pro')
        elif class_index == 1:
            labels.append('neutral')
        elif class_index == 2:
            labels.append('anti')
        
        i += k
        print()
        print()

    all_ents = [pd.NA if ents == set() else ents for ents in all_ents]
    
    df[entity_type] = all_ents
    df[label] = labels
    return df

Stance Annotation

Establishment

In [19]:
data, tracker, sents = batch_nlp(df_articles, 'EST')
df_articles = label_stories(df_articles, 'EST', data, tracker, sents)

df_articles

[Running NER...]:   0%|          | 0/1506 [00:00<?, ?it/s]

[Running sentiments analysis...]


Processing batches: 100%|██████████| 205/205 [02:54<00:00,  1.18batch/s]


('The just-concluded Lok Sabha elections were won by ', 'Narendra Modi', ', not the BJP.')
[0.7778029441833496, 0.19371387362480164, 0.028483174741268158]
('This is evident from the exit poll conducted by Lokniti-CSDS, which shows that one-third of the ', 'BJP', ' voters would have chosen a different party had Modi not been the PM candidate.')
[0.08807327598333359, 0.7539703249931335, 0.15795642137527466]
('Not only are most of the ', 'BJP', ' candidates selected by Modi and Amit Shah, but they depend upon the duo after becoming MPs. Advertisement')
[0.09633024036884308, 0.8489124178886414, 0.054757408797740936]
('Not only are most of the BJP candidates selected by Modi and ', 'Amit Shah', ', but they depend upon the duo after becoming MPs. Advertisement')
[0.10985742509365082, 0.8541414141654968, 0.03600120171904564]
('This transformation of a party which took pride in its collegial and democratic decision-making process is probably more damaging at the state-level, where the ', 'BJP

,id,indexed_date,language,media_name,media_url,publish_date,title,url,text,EST,EST_label
0,040f2d7778040bedacec572f66446e31c51e5cc6ab2fb3...,2025-07-02 15:51:38.113666+00:00,en,indianexpress.com,indianexpress.com,2019-06-21,One man show,https://indianexpress.com/article/opinion/colu...,The just-concluded Lok Sabha elections were wo...,"{BJP, Narendra Modi, Amit Shah}",neutral
1,8c9b2e13a6d8804482a4834e25af9b51c60218e06920d7...,2025-07-02 15:35:31.872823+00:00,en,dnaindia.com,dnaindia.com,2019-06-21,BJP’s audacious target,https://www.dnaindia.com/analysis/column-bjp-s...,Party has set an ambitious goal of dislodging ...,{BJP},pro
2,3ef9d8469c5dc41b1b0ae8875c065132315a5b0c4a4ffe...,2025-07-02 10:24:58.756950+00:00,en,indiatimes.com,indiatimes.com,2019-06-25,"Yesterday, once more: Mayawati faces tough cha...",https://timesofindia.indiatimes.com/blogs/toi-...,Bahujan Samaj Party chief Mayawati yesterday a...,{BJP},pro
3,dcd2a88950e4f78a478eaf612e83fd71e8015e2c2ab43d...,2025-07-02 07:08:09.645774+00:00,en,indiatoday.in,indiatoday.in,2019-06-27,"With Narendra Modi taking lead, how BJP is app...",https://www.indiatoday.in/news-analysis/story/...,"Nationalism, protection of cows from slaughter...","{BJP, Narendra Modi}",pro
4,208f5885d54138a55c2671fed309e2e038c34314885170...,2025-07-02 05:56:16.345701+00:00,en,thehindu.com,thehindu.com,2019-06-28,"Will the idea of ‘one nation, one poll’ work i...",https://www.thehindu.com/opinion/op-ed/will-th...,"Last week, when Prime Minister Narendra Modi c...",{Narendra Modi},unknown
...,...,...,...,...,...,...,...,...,...,...,...
1501,ce0653dfb6ca458d65e587e462ae36e9672f765b21eced...,2023-11-21 00:39:23+00:00,en,indianexpress.com,indianexpress.com,2023-11-19,"P Chidambaram writes: Different states, Differ...",https://indianexpress.com/article/opinion/colu...,Elections are underway in five states — Mizora...,"{BJP, Narendra Modi}",neutral
1502,cdb5a12056dfb4573b7a95e51a6003dc1803506d306512...,2023-11-20 03:26:07+00:00,en,thequint.com,thequint.com,2023-11-18,Telangana Election: What is Congress' Strategy...,https://www.thequint.com/opinion/telangana-ele...,"In 2018, only one BRS Muslim candidate from Bo...",{BJP},unknown
1503,eaabb4162e532903144d582d15b8a8c84a98bcf093fac8...,2023-11-19 02:43:21+00:00,en,hindustantimes.com,hindustantimes.com,2023-11-17,Can INDIA afford to campaign without unifying ...,https://www.hindustantimes.com/opinion/keeping...,The Indian National Developmental Inclusive Al...,"{BJP, Bhartiya Janata Party, Narendra Modi}",anti
1504,fbc8fc2ce95cde01dfcb9e1002b12e751f033f1341a8ee...,2023-11-18 01:03:34+00:00,en,hindustantimes.com,hindustantimes.com,2023-11-16,The battle for central India,https://www.hindustantimes.com/editorials/the-...,Madhya Pradesh and 70 assembly constituencies ...,"{BJP, Shivraj Singh Chouhan}",neutral


Opposition

In [9]:
data, tracker, sents = batch_nlp(df_articles, 'OPP')
df_articles = label_stories(df_articles, 'OPP', data, tracker, sents)
df_articles.to_parquet('elections_opinions_annotated_new.parquet', index=False)

df_articles

[Running NER...]:   0%|          | 0/1506 [00:00<?, ?it/s]

[Running sentiments analysis...]


Processing batches:  52%|█████▏    | 85/165 [01:51<01:51,  1.40s/batch]Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Processing batches: 100%|██████████| 165/165 [03:33<00:00,  1.29s/batch]


('For the BJP, it means that the party depends on one person more than ever before, like the ', 'Congress', ' used to depend on Indira Gandhi in the 1970s-80s.')
[0.08944577723741531, 0.7634824514389038, 0.14707179367542267]

Article Vector: [0.08944578 0.76348245 0.14707179]


('Party has set an ambitious goal of dislodging ', 'Mamata Banerjee', ' from power in 2021 state elections')
[0.02123161032795906, 0.054406359791755676, 0.924362063407898]
('After a record performance in Lok Sabha elections (18 seats / 40% vote share), BJP has set an ambitious target of dislodging ', 'Mamata Banerjee', ' from power in 2021 state elections.')
[0.03699560463428497, 0.07657863944768906, 0.886425793170929]
('Even if TMC and ', 'Congress', ' would have contested together BJP would have still won 12 seats.')
[0.037528567016124725, 0.9170374274253845, 0.045433953404426575]
('The BJP did to TMC, what it did to ', 'Congress', ' in 2014, successfully managing to brand Mamata as anti-Hindu and pro-minorit

,id,indexed_date,language,media_name,media_url,publish_date,title,url,text,EST,EST_label,combined_text,vectors,OPP,OPP_label
0,040f2d7778040bedacec572f66446e31c51e5cc6ab2fb3...,2025-07-02 15:51:38.113666+00:00,en,indianexpress.com,indianexpress.com,2019-06-21,One man show,https://indianexpress.com/article/opinion/colu...,The just-concluded Lok Sabha elections were wo...,"{'BJP', 'Narendra Modi', 'Amit Shah'}",neutral,One man show: The just-concluded Lok Sabha ele...,"[-0.04222811, -0.0115666725, 0.023418004, -0.0...",{Congress},neutral
1,8c9b2e13a6d8804482a4834e25af9b51c60218e06920d7...,2025-07-02 15:35:31.872823+00:00,en,dnaindia.com,dnaindia.com,2019-06-21,BJP’s audacious target,https://www.dnaindia.com/analysis/column-bjp-s...,Party has set an ambitious goal of dislodging ...,{'BJP'},pro,BJP’s audacious target: Party has set an ambit...,"[-0.031283606, 0.014232438, 0.022478845, -0.00...","{Congress, Mamata Banerjee}",neutral
2,3ef9d8469c5dc41b1b0ae8875c065132315a5b0c4a4ffe...,2025-07-02 10:24:58.756950+00:00,en,indiatimes.com,indiatimes.com,2019-06-25,"Yesterday, once more: Mayawati faces tough cha...",https://timesofindia.indiatimes.com/blogs/toi-...,Bahujan Samaj Party chief Mayawati yesterday a...,{'BJP'},pro,"Yesterday, once more: Mayawati faces tough cha...","[-0.043734904, -0.032639347, 0.004154884, 0.01...","{Mayawati, Bahujan Samaj Party, Samajwadi Party}",neutral
3,dcd2a88950e4f78a478eaf612e83fd71e8015e2c2ab43d...,2025-07-02 07:08:09.645774+00:00,en,indiatoday.in,indiatoday.in,2019-06-27,"With Narendra Modi taking lead, how BJP is app...",https://www.indiatoday.in/news-analysis/story/...,"Nationalism, protection of cows from slaughter...","{'BJP', 'Narendra Modi'}",pro,"With Narendra Modi taking lead, how BJP is app...","[-0.01779863, 0.021031821, -0.009818159, -0.00...",{Congress},neutral
4,208f5885d54138a55c2671fed309e2e038c34314885170...,2025-07-02 05:56:16.345701+00:00,en,thehindu.com,thehindu.com,2019-06-28,"Will the idea of ‘one nation, one poll’ work i...",https://www.thehindu.com/opinion/op-ed/will-th...,"Last week, when Prime Minister Narendra Modi c...",{'Narendra Modi'},unknown,"Will the idea of ‘one nation, one poll’ work i...","[-0.058923103, -0.005817032, -0.03389539, -0.0...","{DMK, Tiruchi Siva}",neutral
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1501,ce0653dfb6ca458d65e587e462ae36e9672f765b21eced...,2023-11-21 00:39:23+00:00,en,indianexpress.com,indianexpress.com,2023-11-19,"P Chidambaram writes: Different states, Differ...",https://indianexpress.com/article/opinion/colu...,Elections are underway in five states — Mizora...,"{'BJP', 'Narendra Modi'}",neutral,"P Chidambaram writes: Different states, Differ...","[-0.052230738, -0.008890713, 0.06888017, -0.02...",<NA>,unknown
1502,cdb5a12056dfb4573b7a95e51a6003dc1803506d306512...,2023-11-20 03:26:07+00:00,en,thequint.com,thequint.com,2023-11-18,Telangana Election: What is Congress' Strategy...,https://www.thequint.com/opinion/telangana-ele...,"In 2018, only one BRS Muslim candidate from Bo...",{'BJP'},unknown,Telangana Election: What is Congress' Strategy...,"[-0.05431041, 0.013782984, -0.030381976, -0.00...","{Congress, AIMIM}",neutral
1503,eaabb4162e532903144d582d15b8a8c84a98bcf093fac8...,2023-11-19 02:43:21+00:00,en,hindustantimes.com,hindustantimes.com,2023-11-17,Can INDIA afford to campaign without unifying ...,https://www.hindustantimes.com/opinion/keeping...,The Indian National Developmental Inclusive Al...,"{'BJP', 'Bhartiya Janata Party', 'Narendra Modi'}",anti,Can INDIA afford to campaign without unifying ...,"[-0.05980835, -0.019375194, -0.046474203, -0.0...","{Congress, Rahul Gandhi, Mehbooba Mufti}",neutral
1504,fbc8fc2ce95cde01dfcb9e1002b12e751f033f1341a8ee...,2023-11-18 01:03:34+00:00,en,hindustantimes.com,hindustantimes.com,2023-11-16,The battle for central India,https://www.hindustantimes.com/editorials/the-...,Madhya Pradesh and 70 assembly constituencies ...,"{'BJP', 'Shivraj Singh Chouhan'}",neutral,The battle f

Article Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
df_articles = pd.read_csv('elections_opinions_annotated_new.csv')

df_articles

,id,indexed_date,language,media_name,media_url,publish_date,title,url,text,EST,EST_label
0,040f2d7778040bedacec572f66446e31c51e5cc6ab2fb3...,2025-07-02 15:51:38.113666+00:00,en,indianexpress.com,indianexpress.com,2019-06-21,One man show,https://indianexpress.com/article/opinion/colu...,The just-concluded Lok Sabha elections were wo...,"{'BJP', 'Narendra Modi', 'Amit Shah'}",neutral
1,8c9b2e13a6d8804482a4834e25af9b51c60218e06920d7...,2025-07-02 15:35:31.872823+00:00,en,dnaindia.com,dnaindia.com,2019-06-21,BJP’s audacious target,https://www.dnaindia.com/analysis/column-bjp-s...,Party has set an ambitious goal of dislodging ...,{'BJP'},pro
2,3ef9d8469c5dc41b1b0ae8875c065132315a5b0c4a4ffe...,2025-07-02 10:24:58.756950+00:00,en,indiatimes.com,indiatimes.com,2019-06-25,"Yesterday, once more: Mayawati faces tough cha...",https://timesofindia.indiatimes.com/blogs/toi-...,Bahujan Samaj Party chief Mayawati yesterday a...,{'BJP'},pro
3,dcd2a88950e4f78a478eaf612e83fd71e8015e2c2ab43d...,2025-07-02 07:08:09.645774+00:00,en,indiatoday.in,indiatoday.in,2019-06-27,"With Narendra Modi taking lead, how BJP is app...",https://www.indiatoday.in/news-analysis/story/...,"Nationalism, protection of cows from slaughter...","{'BJP', 'Narendra Modi'}",pro
4,208f5885d54138a55c2671fed309e2e038c34314885170...,2025-07-02 05:56:16.345701+00:00,en,thehindu.com,thehindu.com,2019-06-28,"Will the idea of ‘one nation, one poll’ work i...",https://www.thehindu.com/opinion/op-ed/will-th...,"Last week, when Prime Minister Narendra Modi c...",{'Narendra Modi'},unknown
...,...,...,...,...,...,...,...,...,...,...,...
1501,ce0653dfb6ca458d65e587e462ae36e9672f765b21eced...,2023-11-21 00:39:23+00:00,en,indianexpress.com,indianexpress.com,2023-11-19,"P Chidambaram writes: Different states, Differ...",https://indianexpress.com/article/opinion/colu...,Elections are underway in five states — Mizora...,"{'BJP', 'Narendra Modi'}",neutral
1502,cdb5a12056dfb4573b7a95e51a6003dc1803506d306512...,2023-11-20 03:26:07+00:00,en,thequint.com,thequint.com,2023-11-18,Telangana Election: What is Congress' Strategy...,https://www.thequint.com/opinion/telangana-ele...,"In 2018, only one BRS Muslim candidate from Bo...",{'BJP'},unknown
1503,eaabb4162e532903144d582d15b8a8c84a98bcf093fac8...,2023-11-19 02:43:21+00:00,en,hindustantimes.com,hindustantimes.com,2023-11-17,Can INDIA afford to campaign without unifying ...,https://www.hindustantimes.com/opinion/keeping...,The Indian National Developmental Inclusive Al...,"{'BJP', 'Bhartiya Janata Party', 'Narendra Modi'}",anti
1504,fbc8fc2ce95cde01dfcb9e1002b12e751f033f1341a8ee...,2023-11-18 01:03:34+00:00,en,hindustantimes.com,hindustantimes.com,2023-11-16,The battle for central India,https://www.hindustantimes.com/editorials/the-...,Madhya Pradesh and 70 assembly constituencies ...,"{'BJP', 'Shivraj Singh Chouhan'}",neutral


In [4]:
df_articles['combined_text'] = df_articles['title'].fillna('') + ": " + df_articles['text']

df_articles

,id,indexed_date,language,media_name,media_url,publish_date,title,url,text,EST,EST_label,combined_text
0,040f2d7778040bedacec572f66446e31c51e5cc6ab2fb3...,2025-07-02 15:51:38.113666+00:00,en,indianexpress.com,indianexpress.com,2019-06-21,One man show,https://indianexpress.com/article/opinion/colu...,The just-concluded Lok Sabha elections were wo...,"{'BJP', 'Narendra Modi', 'Amit Shah'}",neutral,One man show: The just-concluded Lok Sabha ele...
1,8c9b2e13a6d8804482a4834e25af9b51c60218e06920d7...,2025-07-02 15:35:31.872823+00:00,en,dnaindia.com,dnaindia.com,2019-06-21,BJP’s audacious target,https://www.dnaindia.com/analysis/column-bjp-s...,Party has set an ambitious goal of dislodging ...,{'BJP'},pro,BJP’s audacious target: Party has set an ambit...
2,3ef9d8469c5dc41b1b0ae8875c065132315a5b0c4a4ffe...,2025-07-02 10:24:58.756950+00:00,en,indiatimes.com,indiatimes.com,2019-06-25,"Yesterday, once more: Mayawati faces tough cha...",https://timesofindia.indiatimes.com/blogs/toi-...,Bahujan Samaj Party chief Mayawati yesterday a...,{'BJP'},pro,"Yesterday, once more: Mayawati faces tough cha..."
3,dcd2a88950e4f78a478eaf612e83fd71e8015e2c2ab43d...,2025-07-02 07:08:09.645774+00:00,en,indiatoday.in,indiatoday.in,2019-06-27,"With Narendra Modi taking lead, how BJP is app...",https://www.indiatoday.in/news-analysis/story/...,"Nationalism, protection of cows from slaughter...","{'BJP', 'Narendra Modi'}",pro,"With Narendra Modi taking lead, how BJP is app..."
4,208f5885d54138a55c2671fed309e2e038c34314885170...,2025-07-02 05:56:16.345701+00:00,en,thehindu.com,thehindu.com,2019-06-28,"Will the idea of ‘one nation, one poll’ work i...",https://www.thehindu.com/opinion/op-ed/will-th...,"Last week, when Prime Minister Narendra Modi c...",{'Narendra Modi'},unknown,"Will the idea of ‘one nation, one poll’ work i..."
...,...,...,...,...,...,...,...,...,...,...,...,...
1501,ce0653dfb6ca458d65e587e462ae36e9672f765b21eced...,2023-11-21 00:39:23+00:00,en,indianexpress.com,indianexpress.com,2023-11-19,"P Chidambaram writes: Different states, Differ...",https://indianexpress.com/article/opinion/colu...,Elections are underway in five states — Mizora...,"{'BJP', 'Narendra Modi'}",neutral,"P Chidambaram writes: Different states, Differ..."
1502,cdb5a12056dfb4573b7a95e51a6003dc1803506d306512...,2023-11-20 03:26:07+00:00,en,thequint.com,thequint.com,2023-11-18,Telangana Election: What is Congress' Strategy...,https://www.thequint.com/opinion/telangana-ele...,"In 2018, only one BRS Muslim candidate from Bo...",{'BJP'},unknown,Telangana Election: What is Congress' Strategy...
1503,eaabb4162e532903144d582d15b8a8c84a98bcf093fac8...,2023-11-19 02:43:21+00:00,en,hindustantimes.com,hindustantimes.com,2023-11-17,Can INDIA afford to campaign without unifying ...,https://www.hindustantimes.com/opinion/keeping...,The Indian National Developmental Inclusive Al...,"{'BJP', 'Bhartiya Janata Party', 'Narendra Modi'}",anti,Can INDIA afford to campaign without unifying ...
1504,fbc8fc2ce95cde01dfcb9e1002b12e751f033f1341a8ee...,2023-11-18 01:03:34+00:00,en,hindustantimes.com,hindustantimes.com,2023-11-16,The battle for central India,https://www.hindustantimes.com/editorials/the-...,Madhya Pradesh and 70 assembly constituencies ...,"{'BJP', 'Shivraj Singh Chouhan'}",neutral,The battle for central India: Madhya Pradesh a...


In [5]:
model = SentenceTransformer('all-MiniLM-L6-v2')
df_articles['vectors'] = list(model.encode(df_articles['combined_text'].tolist(), show_progress_bar=True))

df_articles

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/48 [00:00<?, ?it/s]

,id,indexed_date,language,media_name,media_url,publish_date,title,url,text,EST,EST_label,combined_text,vectors
0,040f2d7778040bedacec572f66446e31c51e5cc6ab2fb3...,2025-07-02 15:51:38.113666+00:00,en,indianexpress.com,indianexpress.com,2019-06-21,One man show,https://indianexpress.com/article/opinion/colu...,The just-concluded Lok Sabha elections were wo...,"{'BJP', 'Narendra Modi', 'Amit Shah'}",neutral,One man show: The just-concluded Lok Sabha ele...,"[-0.04222811, -0.0115666725, 0.023418004, -0.0..."
1,8c9b2e13a6d8804482a4834e25af9b51c60218e06920d7...,2025-07-02 15:35:31.872823+00:00,en,dnaindia.com,dnaindia.com,2019-06-21,BJP’s audacious target,https://www.dnaindia.com/analysis/column-bjp-s...,Party has set an ambitious goal of dislodging ...,{'BJP'},pro,BJP’s audacious target: Party has set an ambit...,"[-0.031283606, 0.014232438, 0.022478845, -0.00..."
2,3ef9d8469c5dc41b1b0ae8875c065132315a5b0c4a4ffe...,2025-07-02 10:24:58.756950+00:00,en,indiatimes.com,indiatimes.com,2019-06-25,"Yesterday, once more: Mayawati faces tough cha...",https://timesofindia.indiatimes.com/blogs/toi-...,Bahujan Samaj Party chief Mayawati yesterday a...,{'BJP'},pro,"Yesterday, once more: Mayawati faces tough cha...","[-0.043734904, -0.032639347, 0.004154884, 0.01..."
3,dcd2a88950e4f78a478eaf612e83fd71e8015e2c2ab43d...,2025-07-02 07:08:09.645774+00:00,en,indiatoday.in,indiatoday.in,2019-06-27,"With Narendra Modi taking lead, how BJP is app...",https://www.indiatoday.in/news-analysis/story/...,"Nationalism, protection of cows from slaughter...","{'BJP', 'Narendra Modi'}",pro,"With Narendra Modi taking lead, how BJP is app...","[-0.01779863, 0.021031821, -0.009818159, -0.00..."
4,208f5885d54138a55c2671fed309e2e038c34314885170...,2025-07-02 05:56:16.345701+00:00,en,thehindu.com,thehindu.com,2019-06-28,"Will the idea of ‘one nation, one poll’ work i...",https://www.thehindu.com/opinion/op-ed/will-th...,"Last week, when Prime Minister Narendra Modi c...",{'Narendra Modi'},unknown,"Will the idea of ‘one nation, one poll’ work i...","[-0.058923103, -0.005817032, -0.03389539, -0.0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1501,ce0653dfb6ca458d65e587e462ae36e9672f765b21eced...,2023-11-21 00:39:23+00:00,en,indianexpress.com,indianexpress.com,2023-11-19,"P Chidambaram writes: Different states, Differ...",https://indianexpress.com/article/opinion/colu...,Elections are underway in five states — Mizora...,"{'BJP', 'Narendra Modi'}",neutral,"P Chidambaram writes: Different states, Differ...","[-0.052230738, -0.008890713, 0.06888017, -0.02..."
1502,cdb5a12056dfb4573b7a95e51a6003dc1803506d306512...,2023-11-20 03:26:07+00:00,en,thequint.com,thequint.com,2023-11-18,Telangana Election: What is Congress' Strategy...,https://www.thequint.com/opinion/telangana-ele...,"In 2018, only one BRS Muslim candidate from Bo...",{'BJP'},unknown,Telangana Election: What is Congress' Strategy...,"[-0.05431041, 0.013782984, -0.030381976, -0.00..."
1503,eaabb4162e532903144d582d15b8a8c84a98bcf093fac8...,2023-11-19 02:43:21+00:00,en,hindustantimes.com,hindustantimes.com,2023-11-17,Can INDIA afford to campaign without unifying ...,https://www.hindustantimes.com/opinion/keeping...,The Indian National Developmental Inclusive Al...,"{'BJP', 'Bhartiya Janata Party', 'Narendra Modi'}",anti,Can INDIA afford to campaign without unifying ...,"[-0.05980835, -0.019375194, -0.046474203, -0.0..."
1504,fbc8fc2ce95cde01dfcb9e1002b12e751f033f1341a8ee...,2023-11-18 01:03:34+00:00,en,hindustantimes.com,hindustantimes.com,2023-11-16,The battle for central India,https://www.hindustantimes.com/editorials/the-...,Madhya Pradesh and 70 assembly constituencies ...,"{'BJP', 'Shivraj Singh Chouhan'}",neutral,The battle for central India: Madhya Pradesh a...,"[-0.0217077, -0.012442368, -0.016473955, -0.03..."


In [7]:
df_articles.to_parquet('elections_opinions_annotated_new.parquet', index=False)

BERTopic Topic Modeling

In [22]:
%uv pip install --upgrade transformers sentence-transformers huggingface_hub openai bertopic

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Using Python 3.10.20 environment at: /Users/rupsharray/Documents/Ashoka/Research/IndiaMediaLens/IndiaMediaLens/.iml_venv
Resolved 61 packages in 1.04s                                        
⠙ Preparing packages... (0/11)                                                  
⠙ Preparing packages... (0/11)------------------     0 B/460.86 KiB          
⠙ Preparing packages... (0/11)------------------ 16.00 KiB/460.86 KiB        
⠙ Preparing packages... (0/11)------------------ 32.00 KiB/460.86 KiB        
⠙ Preparing packages... (0/11)------------------ 48.00 KiB/460.86 KiB        
⠙ Preparing packages... (0/11)------------------ 64.00 KiB/460.86 KiB        
⠙ Preparing pa

In [ ]:
%uv pip install --upgrade transformers sentence-transformers huggingface_hub openai bertopic

In [1]:
import pandas as pd
import openai
from bertopic import BERTopic
from bertopic.representation import OpenAI
from sentence_transformers import SentenceTransformer
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer
import torch

In [2]:
df_articles = pd.read_parquet('elections_opinions_annotated_new.parquet')

df_articles

,id,indexed_date,language,media_name,media_url,publish_date,title,url,text,EST,EST_label,combined_text,vectors,OPP,OPP_label
0,040f2d7778040bedacec572f66446e31c51e5cc6ab2fb3...,2025-07-02 15:51:38.113666+00:00,en,indianexpress.com,indianexpress.com,2019-06-21,One man show,https://indianexpress.com/article/opinion/colu...,The just-concluded Lok Sabha elections were wo...,"{'BJP', 'Narendra Modi', 'Amit Shah'}",neutral,One man show: The just-concluded Lok Sabha ele...,"[-0.04222811, -0.0115666725, 0.023418004, -0.0...",[Congress],neutral
1,8c9b2e13a6d8804482a4834e25af9b51c60218e06920d7...,2025-07-02 15:35:31.872823+00:00,en,dnaindia.com,dnaindia.com,2019-06-21,BJP’s audacious target,https://www.dnaindia.com/analysis/column-bjp-s...,Party has set an ambitious goal of dislodging ...,{'BJP'},pro,BJP’s audacious target: Party has set an ambit...,"[-0.031283606, 0.014232438, 0.022478845, -0.00...","[Congress, Mamata Banerjee]",neutral
2,3ef9d8469c5dc41b1b0ae8875c065132315a5b0c4a4ffe...,2025-07-02 10:24:58.756950+00:00,en,indiatimes.com,indiatimes.com,2019-06-25,"Yesterday, once more: Mayawati faces tough cha...",https://timesofindia.indiatimes.com/blogs/toi-...,Bahujan Samaj Party chief Mayawati yesterday a...,{'BJP'},pro,"Yesterday, once more: Mayawati faces tough cha...","[-0.043734904, -0.032639347, 0.004154884, 0.01...","[Mayawati, Bahujan Samaj Party, Samajwadi Party]",neutral
3,dcd2a88950e4f78a478eaf612e83fd71e8015e2c2ab43d...,2025-07-02 07:08:09.645774+00:00,en,indiatoday.in,indiatoday.in,2019-06-27,"With Narendra Modi taking lead, how BJP is app...",https://www.indiatoday.in/news-analysis/story/...,"Nationalism, protection of cows from slaughter...","{'BJP', 'Narendra Modi'}",pro,"With Narendra Modi taking lead, how BJP is app...","[-0.01779863, 0.021031821, -0.009818159, -0.00...",[Congress],neutral
4,208f5885d54138a55c2671fed309e2e038c34314885170...,2025-07-02 05:56:16.345701+00:00,en,thehindu.com,thehindu.com,2019-06-28,"Will the idea of ‘one nation, one poll’ work i...",https://www.thehindu.com/opinion/op-ed/will-th...,"Last week, when Prime Minister Narendra Modi c...",{'Narendra Modi'},unknown,"Will the idea of ‘one nation, one poll’ work i...","[-0.058923103, -0.005817032, -0.03389539, -0.0...","[DMK, Tiruchi Siva]",neutral
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1501,ce0653dfb6ca458d65e587e462ae36e9672f765b21eced...,2023-11-21 00:39:23+00:00,en,indianexpress.com,indianexpress.com,2023-11-19,"P Chidambaram writes: Different states, Differ...",https://indianexpress.com/article/opinion/colu...,Elections are underway in five states — Mizora...,"{'BJP', 'Narendra Modi'}",neutral,"P Chidambaram writes: Different states, Differ...","[-0.052230738, -0.008890713, 0.06888017, -0.02...",None,unknown
1502,cdb5a12056dfb4573b7a95e51a6003dc1803506d306512...,2023-11-20 03:26:07+00:00,en,thequint.com,thequint.com,2023-11-18,Telangana Election: What is Congress' Strategy...,https://www.thequint.com/opinion/telangana-ele...,"In 2018, only one BRS Muslim candidate from Bo...",{'BJP'},unknown,Telangana Election: What is Congress' Strategy...,"[-0.05431041, 0.013782984, -0.030381976, -0.00...","[Congress, AIMIM]",neutral
1503,eaabb4162e532903144d582d15b8a8c84a98bcf093fac8...,2023-11-19 02:43:21+00:00,en,hindustantimes.com,hindustantimes.com,2023-11-17,Can INDIA afford to campaign without unifying ...,https://www.hindustantimes.com/opinion/keeping...,The Indian National Developmental Inclusive Al...,"{'BJP', 'Bhartiya Janata Party', 'Narendra Modi'}",anti,Can INDIA afford to campaign without unifying ...,"[-0.05980835, -0.019375194, -0.046474203, -0.0...","[Congress, Rahul Gandhi, Mehbooba Mufti]",neutral
1504,fbc8fc2ce95cde01dfcb9e1002b12e751f033f1341a8ee...,2023-11-18 01:03:34+00:00,en,hindustantimes.com,hindustantimes.com,2023-11-16,The battle for central India,https://www.hindustantimes.com/editorials/the-...,Madhya Pradesh and 70 assembly constituencies ...,"{'BJP', 'Shivraj Singh Chouhan'}",neutral,The battle f

In [84]:
# replicability
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', 
                  random_state=42) # random state is the main setting, the rest are defaults needed
                                   # for BERTopic

vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))
# remove english stop words, look at bigrams for topics, + BERTopic default, punctuation removed by default

client = openai.OpenAI(base_url="http://localhost:11434/v1", api_key='ollama')

prompt_label = """
I have a topic that contains the following documents:
[DOCUMENTS]
The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short topic label (not more than 5 words) with appropriate capitalization in the following format:
topic: <topic label>
"""
representation_model = OpenAI(client, model='gemma4:e2b', prompt=prompt_label, generator_kwargs={"reasoning_effort" : "none"})


device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
embeddings = embedding_model.encode(df_articles['combined_text'].to_list(), show_progress_bar=True, batch_size=64)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    vectorizer_model=vectorizer_model,
    #representation_model=representation_model,

    nr_topics=21, # one extra is for misc, will be dropped
    verbose=True
)

Using device: mps


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

In [85]:
topics, _ = topic_model.fit_transform(df_articles['combined_text'].to_list(), embeddings)

2026-04-17 10:17:06,810 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-17 10:17:08,801 - BERTopic - Dimensionality - Completed ✓
2026-04-17 10:17:08,802 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-17 10:17:08,820 - BERTopic - Cluster - Completed ✓
2026-04-17 10:17:08,820 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-04-17 10:17:08,994 - BERTopic - Representation - Completed ✓
2026-04-17 10:17:08,994 - BERTopic - Topic reduction - Reducing number of topics
/Users/rupsharray/Documents/Ashoka/Research/IndiaMediaLens/IndiaMediaLens/.iml_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/rupsharray/Documents/Ashoka/Research/IndiaMediaLens/IndiaMediaLens/.iml_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/rupsharray/

In [86]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,456,-1_bjp_party_elections_modi,"[bjp, party, elections, modi, india, election,...","[On election eve, a tale of two BJPs 20 years ..."
1,0,426,0_bjp_congress_party_seats,"[bjp, congress, party, seats, elections, state...",[Elections 2024: Why Gujarat Continues To Be B...
2,1,108,1_bengal_west bengal_west_mamata,"[bengal, west bengal, west, mamata, tmc, baner...",[Congress routed in 5 assembly polls. Will Rah...
3,2,63,2_gandhi_congress_rahul_party,"[gandhi, congress, rahul, party, rahul gandhi,...",[Sonia takes the wheel as Congress plumbs new ...
4,3,43,3_tamil_nadu_tamil nadu_dmk,"[tamil, nadu, tamil nadu, dmk, aiadmk, stalin,...",[Tamil Nadu Assembly Elections 2021: The Tamil...
5,4,42,4_election_eci_commission_election commission,"[election, eci, commission, election commissio...",[The Election Commission of India was built on...
6,5,40,5_women_women voters_candidates_cases,"[women, women voters, candidates, cases, voter...",[Increased participation to political exclusio...
7,6,40,6_kashmir_jammu_jammu kashmir_union,"[kashmir, jammu, jammu kashmir, union, 370, as...",[Inserting Pakistan into J&K poll talk makes l...
8,7,35,7_modi_election_prime minister_prime,"[modi, election, prime minister, prime, narend...",[Sagarika Ghose writes: Democracy is not in In...
9,8,32,8_bsp_mayawati_dalit_samaj,"[bsp, mayawati, dalit, samaj, bahujan, samaj p...",[Mayawati derails Dalit politics: Is there red...


In [87]:
topic_model.visualize_topics()

In [88]:
topic_model.visualize_heatmap()

In [89]:
hierarchical_topics = topic_model.hierarchical_topics(df_articles['combined_text'].to_list())
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

100%|██████████| 19/19 [00:00<00:00, 855.12it/s]


In [91]:
topic_model.visualize_barchart(top_n_topics=20)

In [2]:
%uv pip install "transformers==4.30.2" "sentence-transformers==2.2.2" "huggingface_hub<0.20.0" "openai==0.28.1"

Using Python 3.10.20 environment at: /Users/rupsharray/Documents/Ashoka/Research/IndiaMediaLens/IndiaMediaLens/.iml_venv
Resolved 44 packages in 1.04s                                        
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)-------------------     0 B/75.15 KiB           
⠙ Preparing packages... (0/1)------------------- 16.00 KiB/75.15 KiB         
⠙ Preparing packages... (0/1)m------------------ 32.00 KiB/75.15 KiB         
⠹ Preparing packages... (0/1)--------------- 39.51 KiB/75.15 KiB         
⠹ Preparing packages... (0/1)--------------- 39.51 KiB/75.15 KiB         
⠹ Preparing packages... (0/1)----------- 48.00 KiB/75.15 KiB         
⠹ Preparing packages... (0/1)---------- 51.82 KiB/75.15 KiB         
⠹ Preparing packages... (0/1)---------- 59.46 KiB/75.15 KiB         
⠹ Preparing packages... (0/1)---------- 75.15 KiB/75.15 KiB         
Prepared 1 package in 218ms                                          

In [1]:
import pandas as pd

df = pd.read_json('articles_with_text.json')
df

,id,indexed_date,language,media_name,media_url,publish_date,title,url,text
0,dad963715e6a61d2481cc08d0aabf80a798d7e215bc36f...,1733081328483,en,ndtv.com,ndtv.com,1578960000000,Court Seeks Poll Body's Response On Plea For V...,https://www.ndtv.com/india-news/delhi-high-cou...,New Delhi: Delhi High Court on Monday directed...
1,a896d451b026e5a2c9e53dc2ea244d54e1d151f98f284c...,1733076288554,en,ndtv.com,ndtv.com,1578960000000,Indian-Origin British MP Inches Closer In Race...,https://www.ndtv.com/indians-abroad/indian-ori...,"Lisa Nandy, the Indian-origin British MP, and ..."
2,7bb1dced01bd427af43b311d3dba663d128695ed76163b...,1732968243987,en,ndtv.com,ndtv.com,1579910400000,"On National Voters' Day, PM Express Gratitude ...",https://www.ndtv.com/india-news/on-national-vo...,New Delhi: National Voters' Day or Rashtriya M...
3,b77aa012d1afa704fc1c449361adb64636ea303025a83f...,1732916929212,en,ndtv.com,ndtv.com,1580256000000,"""Bullet, Not Biryani"" For Anurag Thakur's Detr...",https://www.ndtv.com/india-news/delhi-assembly...,New Delhi: Karnataka Minister CT Ravi has back...
4,ffebb78904fec1139f8da6c90f5350146b3e4fda291d92...,1732804308510,en,ndtv.com,ndtv.com,1581292800000,"Behind Kejriwal’s Hanuman Chalisa Moment, A St...",https://www.ndtv.com/blog/behind-kejriwals-han...,"A few days before Delhi voted, a prominent new..."
...,...,...,...,...,...,...,...,...,...
286,a0e9be68ad6af1388d05b8f88b5fa9f4d3f71d0b09cfd9...,1766891832044,en,hindustantimes.com,hindustantimes.com,1766880000000,A bitter irony: Thackerays seek Muslim support...,https://www.hindustantimes.com/cities/mumbai-n...,"In 1992, the Sena was accused of being involve..."
287,fae5a44d732ff8902dfbe89b5420e974d2aed44981c9d6...,1766729986801,en,hindustantimes.com,hindustantimes.com,1766707200000,Myanmar to hold first general election in 5 yr...,https://www.hindustantimes.com/world-news/myan...,Myanmar will hold the first phase of a general...
288,e0a0b3e590fb6ab28fff1ed51351ccb047d8d433d1dc51...,1766629213711,en,hindustantimes.com,hindustantimes.com,1766620800000,"‘Not like Putin, Zelensky coming together’: BJ...",https://www.hindustantimes.com/cities/mumbai-n...,"Chief minister Devendra Fadnavis, deputy chief..."
289,84c7f7f103aba80019fa26e9b2986ea02d387128d97995...,1766542834097,en,hindustantimes.com,hindustantimes.com,1766534400000,NCP factions close to alliance for Pune civic ...,https://www.hindustantimes.com/cities/pune-new...,Leaders from both the NCP and NCP (SP) held a ...
